In [1]:
import torch 
import torchvision

In [2]:
print("TORCH Version ",torch.__version__)

TORCH Version  2.11.0+cpu


In [3]:
#load the datasets
from torchvision.datasets import MNIST

In [6]:
#download datasets and for train and test use flags train = True/False
data_train = MNIST(root=r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\GitUnTrack\torch_mnist_dataset",
                    download=True,train=True)

data_test = MNIST(root=r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\GitUnTrack\torch_mnist_dataset",
                    download=True,train=False)


In [10]:
#datasets N,img,label
print(type(data_train[0][0])) #PIL image
print(data_train[0][0].size) #Image shape
print(data_train[0][1]) #label

<class 'PIL.Image.Image'>
(28, 28)
5


In [8]:
data_train[0][0]

In [9]:
len(data_train),len(data_test)

(60000, 10000)

In [11]:
#Transform data (Normalization,ToTensor) for ANN 
from torchvision.transforms import Lambda,ToTensor,Compose #Lamda - define custom transform for all elements,Compose - Combines multiple transforms

transform = Compose([
    ToTensor(),
    Lambda(lambda image : image/255),
    Lambda(lambda image : image.view(784))
])

In [12]:
# we can do transform while downloading itself thus save time and space
data_train = MNIST(root=r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\GitUnTrack\torch_mnist_dataset",
                    download=True,train=True,transform = transform)

data_test = MNIST(root=r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\GitUnTrack\torch_mnist_dataset",
                    download=True,train=False,transform= transform)

In [13]:
data_train[0][0].shape

torch.Size([784])

In [14]:
type(data_train[0][0])

torch.Tensor

In [15]:
data_train[0][1]

5

In [16]:
type(data_train[0][1])

int

In [17]:
#Model Build 
from torch import nn,optim

In [35]:
class Model(nn.Module):
    def __init__(self,sizes,batch_size): #sizes = [128,64,10]
        super().__init__()

        self.hidden_layer_1 = nn.Linear(in_features=784, out_features=sizes[0])
        self.activ_1 = nn.Sigmoid()
        self.hidden_layer_2 = nn.Linear(in_features=sizes[0], out_features=sizes[1])
        self.activ_2 = nn.Sigmoid()
        self.output_layer = nn.Linear(in_features=sizes[1], out_features=sizes[2])
        self.activ_3 = nn.Softmax(dim=1)

        self.loss = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.parameters()) #parameters are from super class nn.Module (based on current network definition)
        self.batch_size = batch_size

    def forward(self,inputs):
        x = self.hidden_layer_1(inputs)
        x = self.activ_1(x)
        x = self.hidden_layer_2(x)
        x = self.activ_2(x)
        x = self.output_layer(x)
        x = self.activ_3(x)
        return x 

    def fit(self,X,Y):
        self.optimizer.zero_grad() #initialize weights grads(not weight values) to zero for fresh compute for each epoch
        y_pred = self.forward(X)
        loss = self.loss(y_pred,Y)
        loss.backward() #calculation of gradients - backward propagation 
        self.optimizer.step() #update weights
        return loss.item() #returns only loss values

    def predict(self,X):
        with torch.no_grad():
            return torch.argmax(self.forward(X),axis=1)
    
    def evaluate(self,test_dataloader):
        correct = 0
        for X,Y in test_dataloader:
            Y_pred = self.predict(X)
            correct += (Y == Y_pred ).sum() #batch wise

        accuracy = correct / (len(test_dataloader)*self.batch_size)
        print(f"Accuracy  : {accuracy}")
        

In [36]:
batch_size  = 16
model = Model(sizes=[128,64,10],batch_size = batch_size)

In [25]:
# use data  loader to load datasets as batches to nn (provides func as shuffle)
from torch.utils.data import DataLoader 

In [26]:
train_dataloader = DataLoader(data_train,batch_size = batch_size,shuffle=True)
test_dataloader = DataLoader(data_test,batch_size = batch_size,shuffle=True)

In [38]:
from tqdm import tqdm 

epochs = 10
for i in range(epochs):
    total_loss = 0
    for X,Y in tqdm(train_dataloader,desc=f"Fitting NN : Epoch  : {i+1}"):
        loss = model.fit(X,Y)
        total_loss += loss
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {i+1} Loss :  {avg_loss:.4f}")#loss value should decrease over epoch


Fitting NN : Epoch  : 1: 100%|██████████| 3750/3750 [00:09<00:00, 412.07it/s]


Epoch 1 Loss :  1.6108


Fitting NN : Epoch  : 2: 100%|██████████| 3750/3750 [00:09<00:00, 410.50it/s]


Epoch 2 Loss :  1.6068


Fitting NN : Epoch  : 3: 100%|██████████| 3750/3750 [00:09<00:00, 414.37it/s]


...

Epoch 8 Loss :  1.5878


Fitting NN : Epoch  : 9: 100%|██████████| 3750/3750 [00:08<00:00, 419.07it/s]


Epoch 9 Loss :  1.5848


Fitting NN : Epoch  : 10: 100%|██████████| 3750/3750 [00:09<00:00, 415.95it/s]

Epoch 10 Loss :  1.5822


In [39]:
model.evaluate(test_dataloader)

Accuracy  : 0.8840000033378601


In [42]:
#saving model weights only
save_path = r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\model_weights\mnist_ann_torch_weights.pth"
torch.save(model.state_dict(),save_path)

In [43]:
#loading weights only needs initializing  nn architecture class
model_2 = Model(sizes=[128,64,10],batch_size = 16)
model_2.load_state_dict(torch.load(save_path)) 

<All keys matched successfully>

In [44]:
model_2.evaluate(test_dataloader)

Accuracy  : 0.8840000033378601


In [45]:
#saving model weights + class architecture
save_path_2 =  r"D:\PROJECTS\LEARNINGS\LEARNING_01\MNIST_\model_weights\mnist_ann_torch_full.pth"
torch.save(model,save_path_2)

In [47]:
model_3 = torch.load(save_path_2,weights_only=False)

In [49]:
model_3.evaluate(test_dataloader)

Accuracy  : 0.8840000033378601
